In [2]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# Import libraries
import kagglehub
import os
import pandas as pd
import json


/home/gathoni/Documents/machine-learning/projects/Hospitalflow/cap_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Download the data
path = kagglehub.dataset_download("shalakagangurde/hospital-hmis-dataset-for-healthcare-analytics")

print("Path to dataset files:", path)

# directory = hospital_synthetic_shalaka/hospital_data

Path to dataset files: /home/gathoni/.cache/kagglehub/datasets/shalakagangurde/hospital-hmis-dataset-for-healthcare-analytics/versions/4


In [6]:
csv_files = []

# View the files
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            print(f)
            csv_files.append(os.path.join(root, f))

len(csv_files)

department.csv
admission.csv
billing_detail.csv
patient_insurance.csv
bed.csv
disease.csv
billing.csv
staff_assignment.csv
drug.csv
patient.csv
doctor.csv
employee.csv
diagnostic_test.csv
ward.csv
patient_diagnostic.csv
drug_inventory.csv
drug_manufacturer.csv
insurance_provider.csv
prescription.csv


19

In [7]:

df = pd.read_csv(csv_files[0])
df.head()

,department_id,department_name,department_type,floor_number,status
0,1,Emergency,Clinical,0,Active
1,2,Internal Medicine,Clinical,2,Active
2,3,Surgery,Clinical,3,Active
3,4,Pediatrics,Clinical,2,Active
4,5,Orthopedics,Clinical,4,Active


We are checking for Every column documented with dtype null pct cardinality date range
-> Will go into a data dictionary

In [8]:
details = {}

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    table_name = os.path.splitext(os.path.basename(csv_file))[0]

    columns = df.columns.to_list()

    details[table_name] = {}


    for column in columns:

        # Add types
        dtype_val = str(df.dtypes[column])
        null_pct = float(df[column].isnull().mean() * 100)
        cardinality = df[column].nunique()

        # check if the columns have a date columns
        # can first check the type, and then check if a column has the word date
        date_range = None
        if 'date' in column.lower() or dtype_val == 'datetime':
            column_to_datetime = pd.to_datetime(df[column])
            min_date = column_to_datetime.min()
            max_date = column_to_datetime.max()
            date_range = (str(min_date), str(max_date))
        details[table_name][column]= {
            "dtype": dtype_val,
            "null_pct": null_pct,
            "cardinality": cardinality,
            "date_range": date_range
        }

In [ ]:
details

{'department': {'department_id': {'dtype': 'int64',
   'null_pct': 0.0,
   'cardinality': 11,
   'date_range': None},
  'department_name': {'dtype': 'object',
   'null_pct': 0.0,
   'cardinality': 11,
   'date_range': None},
  'department_type': {'dtype': 'object',
   'null_pct': 0.0,
   'cardinality': 3,
   'date_range': None},
  'floor_number': {'dtype': 'int64',
   'null_pct': 0.0,
   'cardinality': 5,
   'date_range': None},
  'status': {'dtype': 'object',
   'null_pct': 0.0,
   'cardinality': 1,
   'date_range': None}},
 'admission': {'admission_id': {'dtype': 'int64',
   'null_pct': 0.0,
   'cardinality': 45000,
   'date_range': None},
  'admission_date': {'dtype': 'object',
   'null_pct': 0.0,
   'cardinality': 2192,
   'date_range': ('2020-01-01 00:00:00', '2025-12-31 00:00:00')},
  'discharge_date': {'dtype': 'object',
   'null_pct': 0.0,
   'cardinality': 2203,
   'date_range': ('2020-01-02 00:00:00', '2026-01-12 00:00:00')},
  'admission_type': {'dtype': 'object',
   'null_p

In [10]:
# Flatten the nested dictionary into a DataFrame
rows = []
for table_name, columns in details.items():
    for column_name, column_info in columns.items():
        rows.append({
            'table': table_name,
            'column': column_name,
            'dtype': column_info['dtype'],
            'null_pct': column_info['null_pct'],
            'cardinality': column_info['cardinality'],
            'date_range': column_info['date_range']
        })

df_summary = pd.DataFrame(rows)
df_summary.to_csv('../data/raw/data_dictionary_summary.csv', index=False)

In [11]:
# Saved to json too for easy viewing
with open('../data/raw/data_dictionary_summary.json', 'w') as f:
    json.dump(details, f, indent=2, default=str)